# Topic 2: Prompt Template
---
Foundation: what's a prompt?


A prompt is just text you send to the LLM. But it has to be structured: it tells the model who it is, what rules to follow, what context to use, and what to answer. A PromptTemplate is a reusable recipe — a string with {placeholders} that you fill in with real values each time.
The whole idea: you define the structure once, then substitute different context/question values per call. No string concatenation, no copy-paste.
The two slots that matter for RAG
```
prompt = PromptTemplate.from_template(
    "Answer using only this context.\n"
    "Context:\n{context}\n\n"
    "Question: {question}\n"
    "Answer:"
)
```
- {context} → will be filled with your retrieved, joined chunks
- {question} → will be filled with the user's question

```
$ cd /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain && timeout 120 /home/vanitas/Python/git/elusive_Agentic_AI/venv/bin/python - <<'EOF' 2>&1 | grep -v DeprecationWarning | grep -v "fitz"
from langchain_core.prompts import PromptTemplate

# --- STEP 1: define the template once (placeholders in {}) ---
template = """You are a helpful assistant that answers from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

# --- STEP 2: fill it in (this is pure string substitution) ---
filled = prompt.format(
    context="[chunk 1] cover up all the shame like Adam and Eve",
    question="what is paranoid about?",
)

print(filled)
EOF
You are a helpful assistant that answers from the provided context.
If the answer is not in the context, say "I don't know."

Context:
[chunk 1] cover up all the shame like Adam and Eve

Question: what is paranoid about?

Answer:
```

That's the core of Topic 2 — a template, filled. prompt.format(...) replaces {context} and {question} with whatever you pass.


What makes a good RAG prompt
The prompt isn't just structure — the instructions you write are what make answers grounded vs. hallucinated. Three critical rules, each of which you should write into the template:


1. "Answer only from the context" → stops the model from making things up outside your data.
2. "If not in context, say 'I don't know'" → gives it an honest escape hatch instead of guessing.
3. "Quote the context where relevant" → forces it to cite / stay faithful.

```
$ cd /home/vanitas/Python/git/elusive_Agentic_AI/section_5_LangChain && timeout 200 /home/vanitas/Python/git/elusive_Agentic_AI/venv/bin/python - <<'EOF' 2>&1 | grep -v DeprecationWarning | grep -v "fitz"
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="llama3:8b")
parser = StrOutputParser()

# context that does NOT contain the answer
context = "The album eve is about running, exile, and hiding from yourself."
question = "What color are the carpets in the album?"

# PROMPT A: no grounding instructions -> model free to guess
prompt_a = PromptTemplate.from_template(
    "Question: {question}\nAnswer:"
)

# PROMPT B: grounding instructions -> must admit ignorance
prompt_b = PromptTemplate.from_template(
    "Answer ONLY from the context. If not present, say \"I don't know.\" \n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

print("===== PROMPT A (no guard) — asks LLM directly =====")
a = (prompt_a | llm | parser).invoke({"question": question})
print(a.strip())
print()

print("===== PROMPT B (grounded) — gives context empty of answer =====")
b = (prompt_b | llm | parser).invoke({"context": context, "question": question})
print(b.strip())
EOF
===== PROMPT A (no guard) — asks LLM directly =====
I'm happy to help! However, I don't see an album mentioned. Could you please provide more context or clarify which album you're referring to? Once I know which album you're talking about, I'll do my best to answer your question about the color of the carpets.

===== PROMPT B (grounded) — gives context empty of answer =====
I don't know. The context only mentions themes like running, exile, and hiding from oneself, but doesn't mention anything about carpet colors.
```

That demo shows the whole point of Topic 2. With Prompt A (no context at all), the model dodges but would happily guess if it had any pretext. With Prompt B (grounded + guard), it correctly says "I don't know" rather than making up an answer. The prompt template is where you encode truthfulness — not the model.


Two variants you'll run into
1. from_template — you just saw it. Auto-detects {placeholders}.
2. ChatPromptTemplate — for chat models, with roles (system/human). More realistic for production:
```
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer only from the context: {context}"),
    ("human", "{question}"),
])
```
PromptTemplate (plain) is enough to understand the concept; ChatPromptTemplate is what you'd see in the wild with ChatOllama. We'll use the plain one for teaching clarity.
One thing to internalize
The prompt template does pure string substitution — no AI, no vectors. It's just a well-designed f-string with a job: take retrieved context + user question, and lay them out so the LLM behaves. Getting the template right is often the difference between a grounded system and a hallucinating one.

---


Topic 2 complete — the summary
- A PromptTemplate is a reusable recipe with {placeholders}.
- prompt.format(context=..., question=...) fills it in. No magic.
- The instructions in the template control model behavior:
- "answer only from context" → grounded
- "say 'I don't know' if absent" → honest
- "quote the context" → faithful
- We proved it: same question, with vs. without the guard → model correctly admits ignorance when the guard is present.
- Two forms: PromptTemplate (plain) and ChatPromptTemplate (roles for chat models).

In [ ]:
# ---- Step 1: define the prompt template ONCE (placeholders in {}) ----
from langchain_core.prompts import PromptTemplate

template = """You are a helpful assistant that answers from the provided context.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate.from_template(template)

# ---- Step 2: fill it in (pure string substitution, no AI involved) ----
filled = prompt.format(
    context="[chunk 1] cover up all the shame like Adam and Eve",
    question="what is paranoid about?",
)
print(filled)

In [ ]:
# ---- Why the instructions matter: grounded vs. hallucinated ----
# Same question, with and without the grounding guard. The context does NOT
# contain the answer, so a well-built prompt must say "I don't know".
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:8b")
parser = StrOutputParser()

context = "The album eve is about running, exile, and hiding from yourself."
question = "What color are the carpets in the album?"

# PROMPT A: no grounding instructions -> the model is free to guess
prompt_a = PromptTemplate.from_template("Question: {question}\nAnswer:")

# PROMPT B: grounding + escape hatch -> must admit ignorance
prompt_b = PromptTemplate.from_template(
    "Answer ONLY from the context. If not present, say \"I don't know.\" \n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

print("===== PROMPT A (no guard) =====")
a = (prompt_a | llm | parser).invoke({"question": question})
print(a.strip())
print()

print("===== PROMPT B (grounded) =====")
b = (prompt_b | llm | parser).invoke({"context": context, "question": question})
print(b.strip())

In [ ]:
# ---- The variant you will see in production: ChatPromptTemplate ----
# Adds an explicit ROLE per message (system vs human).
from langchain_core.prompts import ChatPromptTemplate

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You answer only from the context: {context}"),
    ("human", "{question}"),
])
print(chat_prompt)